In [1]:
import os
from collections import defaultdict
os.environ["KERAS_BACKEND"] = "jax"
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_hub
import tensorflow as tf
from keras import ops
import keras_rs

e:\COOLYEAH\skripsi\project-code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pickle

In [3]:
users = pd.read_csv('data/users/all_users.csv')
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 1798 entries, 0 to 1797
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0.5  1798 non-null   int64  
 1   Unnamed: 0.4  1798 non-null   int64  
 2   Unnamed: 0.3  1798 non-null   int64  
 3   Unnamed: 0.2  1798 non-null   int64  
 4   Unnamed: 0.1  1798 non-null   int64  
 5   Unnamed: 0    1798 non-null   int64  
 6   ID Anggota    1798 non-null   int64  
 7   prodi_id      1786 non-null   str    
 8   jurusan       1786 non-null   str    
 9   fakultas_id   1786 non-null   float64
 10  fakultas      1786 non-null   str    
 11  jenjang       1786 non-null   str    
 12  Nama Anggota  1798 non-null   str    
 13  role          1798 non-null   str    
 14  angkatan      1774 non-null   float64
 15  userId        1798 non-null   int64  
dtypes: float64(2), int64(8), str(6)
memory usage: 359.6 KB


In [4]:
books = pd.read_csv('data/processed/books_enriched.csv', sep=';')
books.info()

<class 'pandas.DataFrame'>
RangeIndex: 14502 entries, 0 to 14501
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   kode_buku                 14502 non-null  str    
 1   isbn_issn                 12794 non-null  str    
 2   judul_buku                14502 non-null  str    
 3   subjudul                  7818 non-null   str    
 4   penulis                   13490 non-null  str    
 5   nama_belakang             13838 non-null  str    
 6   topik                     13806 non-null  str    
 7   deskripsi                 14502 non-null  str    
 8   bahasa                    14502 non-null  str    
 9   kategori                  14502 non-null  str    
 10  tahun_terbit              14305 non-null  float64
 11  key                       14502 non-null  str    
 12  confidence                14257 non-null  float64
 13  volume_info               13681 non-null  str    
 14  isbn_10_enrich   

In [5]:
class SasRecLLM(keras.Model):
    def __init__(
        self,
        vocabulary_size,
        num_layers,
        num_heads,
        hidden_dim,
        llm_embedding_matrix,
        dropout=0.0,
        max_sequence_length=100,
        dtype=None,
        k=10,
        **kwargs,
    ):
        super().__init__(dtype=dtype, **kwargs)
        
        self.llm_dim = llm_embedding_matrix.shape[1]

        # ======== Layers ========
        self.item_embedding = keras.layers.Embedding(
            input_dim=vocabulary_size,
            output_dim=self.llm_dim,
            embeddings_initializer=keras.initializers.Constant(llm_embedding_matrix),
            trainable=False, # freeze
            dtype=dtype,
            name="item_embedding",
        )
        
        # === 2. Projection Layer ===
        self.projection = keras.layers.Dense(
            units=hidden_dim,
            activation=None,
            name="llm_projection"
        ) if self.llm_dim != hidden_dim else None

        # === 3. Position Embeddings ===
        self.position_embedding = keras_hub.layers.PositionEmbedding(
            initializer="glorot_uniform",
            sequence_length=max_sequence_length,
            dtype=dtype,
            name="position_embedding",
        )
        self.embeddings_add = keras.layers.Add(dtype=dtype, name="embeddings_add")
        self.embeddings_dropout = keras.layers.Dropout(dropout, dtype=dtype, name="embeddings_dropout")

        # === Decoder layers ===
        self.transformer_layers = []
        for i in range(num_layers):
            self.transformer_layers.append(
                keras_hub.layers.TransformerDecoder(
                    intermediate_dim=hidden_dim,
                    num_heads=num_heads,
                    dropout=dropout,
                    layer_norm_epsilon=1e-05,
                    activation="relu",
                    kernel_initializer="glorot_uniform",
                    normalize_first=True,
                    dtype=dtype,
                    name=f"transformer_layer_{i}",
                )
            )

        self.layer_norm = keras.layers.LayerNormalization(axis=-1, epsilon=1e-8, dtype=dtype, name="layer_norm")

        # === Loss ===
        self.loss_fn = keras.losses.BinaryCrossentropy(from_logits=True, reduction=None)

        # === Attributes ===
        self.vocabulary_size = vocabulary_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.max_sequence_length = max_sequence_length

    def _get_last_non_padding_token(self, tensor, padding_mask):
        valid_token_mask = ops.logical_not(padding_mask)
        seq_lengths = ops.sum(ops.cast(valid_token_mask, "int32"), axis=1)
        last_token_indices = ops.maximum(seq_lengths - 1, 0)
        indices = ops.expand_dims(last_token_indices, axis=(-2, -1))
        gathered_tokens = ops.take_along_axis(tensor, indices, axis=1)
        return ops.squeeze(gathered_tokens, axis=1)

    def call(self, inputs, training=False):
        item_ids, padding_mask = inputs["item_ids"], inputs["padding_mask"]

        x = self.item_embedding(item_ids)
        
        if self.projection is not None:
            x = self.projection(x)
            
        position_embedding = self.position_embedding(x)
        x = self.embeddings_add((x, position_embedding))
        x = self.embeddings_dropout(x, training=training)

        for transformer_layer in self.transformer_layers:
            x = transformer_layer(x, decoder_padding_mask=padding_mask, training=training)

        item_sequence_embedding = self.layer_norm(x)
        result = {"item_sequence_embedding": item_sequence_embedding}

        # Inference
        if not training:
            last_item_embedding = self._get_last_non_padding_token(item_sequence_embedding, padding_mask)
            
            all_items = ops.arange(0, self.vocabulary_size)
            all_item_embs = self.item_embedding(all_items)
            if self.projection is not None:
                all_item_embs = self.projection(all_item_embs)
                
            scores = ops.matmul(last_item_embedding, ops.transpose(all_item_embs))
            
            _, top_k_indices = ops.top_k(scores, k=20) 
            result["predictions"] = top_k_indices

        return result

    def compute_loss(self, x, y, y_pred, sample_weight, training=False):
        item_sequence_embedding = y_pred["item_sequence_embedding"]
        y_positive_sequence = y["positive_sequence"]
        y_negative_sequence = y["negative_sequence"]

        pos_emb = self.item_embedding(y_positive_sequence)
        neg_emb = self.item_embedding(y_negative_sequence)
        
        if self.projection is not None:
            pos_emb = self.projection(pos_emb)
            neg_emb = self.projection(neg_emb)

        # Logits
        positive_logits = ops.sum(ops.multiply(pos_emb, item_sequence_embedding), axis=-1)
        negative_logits = ops.sum(ops.multiply(neg_emb, item_sequence_embedding), axis=-1)
        logits = ops.concatenate([positive_logits, negative_logits], axis=1)

        # Labels & Weights
        labels = ops.concatenate([ops.ones_like(positive_logits), ops.zeros_like(negative_logits)], axis=1)
        sample_weight = ops.concatenate([sample_weight, sample_weight], axis=1)

        loss = self.loss_fn(
            y_true=ops.expand_dims(labels, axis=-1),
            y_pred=ops.expand_dims(logits, axis=-1),
            sample_weight=sample_weight,
        )
        return ops.divide_no_nan(ops.sum(loss), ops.sum(sample_weight))

In [6]:
with open('saved_model/recsys_artifacts.pkl', 'rb') as f:
    artifacts = pickle.load(f)

id_to_books = artifacts['id_to_books']
id_to_users = artifacts['id_to_users']
sequences_dict = artifacts['sequences_dict']
item_embeddings = np.load('saved_model/item_embeddings.npy')
NUM_ITEMS = len(id_to_books)

print(f"Artifacts loaded: {NUM_ITEMS} items, {len(sequences_dict)} user sequences.")

Artifacts loaded: 14503 items, 1799 user sequences.


In [7]:
model = SasRecLLM(
    vocabulary_size=NUM_ITEMS,
    num_layers=2,
    num_heads=2,
    hidden_dim=256,
    llm_embedding_matrix=item_embeddings,
    max_sequence_length=10
)

# Build model with dummy input
dummy_ids = np.zeros((1, 10), dtype=np.int32)
dummy_mask = np.ones((1, 10), dtype=bool)
model({"item_ids": dummy_ids, "padding_mask": dummy_mask}, training=False)

# Load the best weights
weights_path = 'saved_model/best_sasrec_weights.weights.h5'
model.load_weights(weights_path, skip_mismatch=True)
print("Model weights loaded successfully.")

Model weights loaded successfully.


In [8]:
def predict_for_user_sog(user_id, model, sequences_dict, max_len, item_embeddings, 
                        sog_weights=(0.7, 0.1, 0.1, 0.1), top_k=10, pool_size=100):
    if user_id not in sequences_dict:
        return None, []

    raw_seq = [int(item.get('bookId', item.get('book_id'))) for item in sequences_dict[user_id]]
    
    seq_truncated = raw_seq[-max_len:]
    item_ids = np.zeros(max_len, dtype="int32")
    item_ids[:len(seq_truncated)] = seq_truncated

    input_ids = tf.convert_to_tensor([item_ids])
    input_mask = tf.convert_to_tensor([item_ids == 0])
    outputs = model({"item_ids": input_ids, "padding_mask": input_mask}, training=False)
    
    last_idx = len(seq_truncated) - 1
    query_vec = outputs["item_sequence_embedding"][0, last_idx, :]

    if hasattr(model, 'projection') and model.projection is not None:
        item_embs_projected = model.projection(item_embeddings)
    else:
        item_embs_projected = item_embeddings
        
    scores = tf.matmul(tf.expand_dims(query_vec, 0), item_embs_projected, transpose_b=True)
    scores = scores.numpy()[0]

    history_set = set(raw_seq)
    ranked_indices = np.argsort(-scores)
    
    pool_indices, pool_relevance = [], []
    for idx in ranked_indices:
        if idx != 0 and idx not in history_set:
            pool_indices.append(idx)
            pool_relevance.append(scores[idx])
        if len(pool_indices) >= pool_size:
            break
    
    if not pool_indices: return [], raw_seq
    
    candidate_items = np.array(pool_indices)
    if hasattr(item_embs_projected, 'numpy'):
        item_embs_projected = item_embs_projected.numpy()
    candidate_vecs = item_embs_projected[candidate_items]
    valid_hist_ids = [int(i) for i in raw_seq if int(i) != 0]
    history_vecs = item_embs_projected[valid_hist_ids]
    
    def cos_sim(a, b):
        a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
        b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
        return a_n @ b_n.T

    rel = (pool_relevance - np.min(pool_relevance)) / (np.max(pool_relevance) - np.min(pool_relevance) + 1e-8)
    sim_to_hist = cos_sim(candidate_vecs, history_vecs)
    dis = 1.0 - np.mean(sim_to_hist, axis=1)
    sim_to_cand = cos_sim(candidate_vecs, candidate_vecs)
    div = 1.0 - np.mean(sim_to_cand, axis=1)
    
    w_rel, w_div, w_dis, w_unpop = sog_weights
    serendipity_scores = (w_rel * rel) + (w_div * div) + (w_dis * dis)
    
    top_indices = candidate_items[np.argsort(-serendipity_scores)[:top_k]]
    return top_indices, raw_seq

In [9]:
id_to_users.get(22082010145)

In [10]:
def get_recommendation_for_user(id_input, sog_weights=(0.7, 0.1, 0.1, 0.1)):
    
    result = users[users['ID Anggota'].astype(int) == int(id_input)]
    
    if result.empty:
        return f"Error: ID Anggota '{id_input}' tidak ditemukan di database user."
    
    idx_row = float(result.index[0])
    print(f'userId : {idx_row}')
    
    if idx_row is None:
        return f"Error: ID Anggota '{id_input}' tidak ditemukan di database user."
    
    if idx_row not in sequences_dict:
        return f"Error: ID Anggota '{id_input}' ditemukan, tapi tidak memiliki riwayat peminjaman."

    preds, hist = predict_for_user_sog(
        user_id=idx_row + 1,
        model=model,
        sequences_dict=sequences_dict,
        max_len=10,
        item_embeddings=item_embeddings,
        sog_weights=sog_weights,
        top_k=10
    )

    print(f"--- Rekomendasi untuk User: {id_input} ---")
    print("\nBuku terakhir yang dipinjam:")
    for i, bid_int in enumerate(hist[-5:]):
        book_code = id_to_books.get(bid_int, "N/A")
        book_title = books[books['book_id'] == book_code]['judul_buku']
        print(f"{i+1}. {book_code} - {book_title}")

    print("\nTop 10 Rekomendasi (SASRec + SOG):")
    for i, bid_int in enumerate(preds):
        book_code = id_to_books.get(bid_int, "N/A")
        book_title = books[books['book_id'] == book_code]['judul_buku']
        print(f"{i+1}. {book_code} - {book_title}")
    
    return preds

In [11]:
[int(item.get('bookId', item.get('book_id'))) for item in sequences_dict[677]]

[12347, 13947, 12347, 14246, 12347, 14246, 14246, 12119, 12396, 12831]

In [12]:
get_recommendation_for_user(24013010088)

userId : 1177.0
--- Rekomendasi untuk User: 24013010088 ---

Buku terakhir yang dipinjam:
1. B018157 - 12163    praktikum  audit  buku 1
Name: judul_buku, dtype: str
2. B024533 - 13878    auditing
Name: judul_buku, dtype: str
3. B024643 - 13906    dasar-dasar manajemen keuangan (buku 2)
Name: judul_buku, dtype: str
4. B024534 - 13879    auditing,  jilid 2
Name: judul_buku, dtype: str
5. B024533 - 13878    auditing
Name: judul_buku, dtype: str

Top 10 Rekomendasi (SASRec + SOG):
1. B014402 - 10115    Metode statistika untuk penelitian kuantitatif
Name: judul_buku, dtype: str
2. B003283 - 1416    Statistik Untuk Bisnis & Ekonomi
Name: judul_buku, dtype: str
3. B024204 - 13809    Metode statistika untuk bisnis dan ekonomi
Name: judul_buku, dtype: str
4. B022822 - 13530    Pasar tenaga kerja Indonesia
Name: judul_buku, dtype: str
5. B027521 - 14344    statistika deskriptif  untuk ekonomi dan bisnis
Name: judul_buku, dtype: str
6. B019704 - 12657    statistika untuk ekonomi dan keuangan mod

array([10116,  1417, 13810, 13531, 14345, 12658, 14374, 14272, 13983,
       10109])

In [14]:
get_recommendation_for_user(24013010057)

userId : 1172.0
--- Rekomendasi untuk User: 24013010057 ---

Buku terakhir yang dipinjam:
1. B018201 - 12177    Sistem Akuntansi, Edisi 4
Name: judul_buku, dtype: str
2. B018038 - 12131    Etika bisnis dan profesi
Name: judul_buku, dtype: str
3. B025477 - 14075    akuntansi keuangan lanjutan di indonesia, buku 2
Name: judul_buku, dtype: str
4. B024533 - 13878    auditing
Name: judul_buku, dtype: str
5. B026342 - 14232    Akuntansi biaya
Name: judul_buku, dtype: str

Top 10 Rekomendasi (SASRec + SOG):
1. B014402 - 10115    Metode statistika untuk penelitian kuantitatif
Name: judul_buku, dtype: str
2. B003283 - 1416    Statistik Untuk Bisnis & Ekonomi
Name: judul_buku, dtype: str
3. B022822 - 13530    Pasar tenaga kerja Indonesia
Name: judul_buku, dtype: str
4. B024204 - 13809    Metode statistika untuk bisnis dan ekonomi
Name: judul_buku, dtype: str
5. B027899 - 14373    metodologi penelitian
Name: judul_buku, dtype: str
6. B027521 - 14344    statistika deskriptif  untuk ekonomi dan bis

array([10116,  1417, 13531, 13810, 14374, 14345, 12173, 14272,  9875,
       12658])

In [15]:
get_recommendation_for_user(24071010088)

userId : 1478.0
--- Rekomendasi untuk User: 24071010088 ---

Buku terakhir yang dipinjam:
1. B020742 - 13038    Memberantas korupsi bersama KPK, Komisi Pember...
Name: judul_buku, dtype: str
2. B013300 - 9585    The Tokyo Trial:War Criminals and Japan’s Post...
Name: judul_buku, dtype: str

Top 10 Rekomendasi (SASRec + SOG):
1. B022822 - 13530    Pasar tenaga kerja Indonesia
Name: judul_buku, dtype: str
2. B027899 - 14373    metodologi penelitian
Name: judul_buku, dtype: str
3. B013865 - 9874    bela negara
Name: judul_buku, dtype: str
4. B003283 - 1416    Statistik Untuk Bisnis & Ekonomi
Name: judul_buku, dtype: str
5. B025065 - 13981    Hukum bisnis
Name: judul_buku, dtype: str
6. B018191 - 12172    manajemen ritel
Name: judul_buku, dtype: str
7. B014402 - 10115    Metode statistika untuk penelitian kuantitatif
Name: judul_buku, dtype: str
8. B027851 - 14371    filsafat pancasila
Name: judul_buku, dtype: str
9. B026460 - 14271    metode penelitian administrasi publik
Name: judul_buku

array([13531, 14374,  9875,  1417, 13982, 12173, 10116, 14372, 14272,
       13810])